In [1]:
# Stage 1: Download reference files
from pathlib import Path
import urllib.request

DATA_DIR = Path(".")

BASE_URL = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage"
SOURCES = {
    "pedigree":   f"{BASE_URL}/20130606_g1k_3202_samples_ped_population.txt",
    "index_2504": f"{BASE_URL}/1000G_2504_high_coverage.sequence.index",
    "index_698":  f"{BASE_URL}/1000G_698_related_high_coverage.sequence.index",
}

paths = {}
for name, url in SOURCES.items():
    dest = DATA_DIR / Path(url).name
    if dest.exists():
        print(f"[cached]   {dest.name} ({dest.stat().st_size:,} bytes)")
    else:
        print(f"[download] {url}")
        urllib.request.urlretrieve(url, dest)
        print(f"           -> {dest} ({dest.stat().st_size:,} bytes)")
    paths[name] = dest

[cached]   20130606_g1k_3202_samples_ped_population.txt (97,483 bytes)
[cached]   1000G_2504_high_coverage.sequence.index (982,480 bytes)
[cached]   1000G_698_related_high_coverage.sequence.index (294,388 bytes)


In [ ]:
import pandas as pd
# Stage 1b: Inspect the pedigree
ped = pd.read_csv(paths["pedigree"], sep=r"\s+")
print(f"Shape: {ped.shape}")
print(f"Columns: {list(ped.columns)}")
ped.head()

Shape: (3202, 7)
Columns: ['FamilyID', 'SampleID', 'FatherID', 'MotherID', 'Sex', 'Population', 'Superpopulation']


,FamilyID,SampleID,FatherID,MotherID,Sex,Population,Superpopulation
0,HG00096,HG00096,0,0,1,GBR,EUR
1,HG00097,HG00097,0,0,2,GBR,EUR
2,HG00099,HG00099,0,0,2,GBR,EUR
3,HG00100,HG00100,0,0,2,GBR,EUR
4,HG00101,HG00101,0,0,1,GBR,EUR


In [3]:
# Stage 1c: Superpopulation and population breakdown
print("Superpopulations:")
print(ped["Superpopulation"].value_counts(dropna=False))
print(f"\n# unique populations: {ped['Population'].nunique()}")
print(ped["Population"].value_counts(dropna=False))

Superpopulations:
Superpopulation
AFR    893
EUR    633
SAS    601
EAS    585
AMR    490
Name: count, dtype: int64

# unique populations: 26
Population
CEU    179
GWD    178
YRI    178
CHS    163
IBS    157
ESN    149
PJL    146
PUR    139
CLM    132
BEB    131
PEL    122
KHV    122
ACB    116
STU    114
ITU    107
TSI    107
JPT    104
CHB    103
GIH    103
FIN     99
MSL     99
LWK     99
MXL     97
CDX     93
GBR     91
ASW     74
Name: count, dtype: int64


In [4]:
# Stage 1d: Sanity checks on the pedigree structure
print(f"Unique SampleIDs:    {ped['SampleID'].nunique()} (should equal {len(ped)})")
print(f"Unique FamilyIDs:    {ped['FamilyID'].nunique()}")
print(f"Founders (no parents): {((ped['FatherID']=='0') & (ped['MotherID']=='0')).sum()}")
print(f"Sex values:          {ped['Sex'].value_counts().to_dict()}  # 1=M, 2=F per PED spec")

# How many family-IDs have 1 / 2 / 3 / 4+ members?
print("\nFamily size distribution:")
print(ped["FamilyID"].value_counts().value_counts().sort_index())

Unique SampleIDs:    3202 (should equal 3202)
Unique FamilyIDs:    1879
Founders (no parents): 2590
Sex values:          {2: 1603, 1: 1599}  # 1=M, 2=F per PED spec

Family size distribution:
count
1    1199
2     106
3     547
4       6
6      21
Name: count, dtype: int64


In [23]:
# Stage 2: Build trios

ped = pd.read_csv(paths["pedigree"], sep=r"\s+", dtype=str)
ped = ped.apply(lambda c: c.str.strip())

sample_ids = set(ped["SampleID"])

trios = {}
n_duos = 0

SPECHLA_POP_MAP = {
    "EAS": "Asian",
    "SAS": "Asian",
    "AFR": "Black",
    "EUR": "Caucasian",
    "AMR": "Unknown",
}

for _, row in ped.iterrows():
    fid, sid, pid, mid = row["FamilyID"], row["SampleID"], row["FatherID"], row["MotherID"]

    if pid == "0" and mid == "0":
        continue

    if pid == "0" or mid == "0" or pid not in sample_ids or mid not in sample_ids:
        n_duos += 1
        continue

    key = f"{fid}_{pid}_{mid}_{sid}"
    if key in trios:
        raise ValueError(f"Duplicate trio key: {key}")

    trios[key] = {
        "family_id": fid,
        "population": row["Population"],
        "superpopulation": row["Superpopulation"],
        "spechla_pop": SPECHLA_POP_MAP[row["Superpopulation"]],
        "members": {
            "child":  {"sample_id": sid, "sex": "M" if row["Sex"] == "1" else "F"},
            "father": {"sample_id": pid, "sex": "M"},
            "mother": {"sample_id": mid, "sex": "F"},
        },
    }

print(f"Trios: {len(trios)}")
print(f"Duos:  {n_duos}")

Trios: 602
Duos:  10


In [22]:
# Stage 2: Build trios

SPECHLA_POP_MAP = {
    "EAS": "Asian",
    "SAS": "Asian",
    "AFR": "Black",
    "EUR": "Caucasian",
    "AMR": "Unknown",
}

with open(paths["pedigree"]) as f:
    header = f.readline().strip().split()
    header_cols_stripped = [h.strip() for h in header]
    col = {name: idx for idx, name in enumerate(header_cols_stripped)}
    rows = [line.strip().split() for line in f if line.strip()]
    for r in rows:
        if len(r) != len(header):
            raise ValueError(f"Malformed row (got {len(r)} fields, expected {len(header)}): {r}")

sample_ids = {row[col["SampleID"]] for row in rows}

trios = {}
n_duos = 0

for row in rows:
    fid = row[col["FamilyID"]].strip()
    sid = row[col["SampleID"]].strip()
    pid = row[col["FatherID"]].strip()
    mid = row[col["MotherID"]].strip()
    sex = row[col["Sex"]].strip()
    pop = row[col["Population"]].strip()
    spop = row[col["Superpopulation"]].strip()

    if pid == "0" and mid == "0":
        continue

    if pid == "0" or mid == "0" or pid not in sample_ids or mid not in sample_ids:
        n_duos += 1
        continue

    key = f"{fid}_{pid}_{mid}_{sid}"
    if key in trios:
        raise ValueError(f"Duplicate trio key: {key}")

    trios[key] = {
        "family_id": fid,
        "population": pop,
        "superpopulation": spop,
        "spechla_pop": SPECHLA_POP_MAP[spop],
        "members": {
            "child":  {"sample_id": sid, "sex": "M" if sex == "1" else "F"},
            "father": {"sample_id": pid, "sex": "M"},
            "mother": {"sample_id": mid, "sex": "F"},
        },
    }

print(f"Trios: {len(trios)}")
print(f"Duos:  {n_duos}")

Trios: 602
Duos:  10


In [ ]:
import urllib.request
import urllib.error

cram_index = {}

n_duplicates = 0

for path in [paths["index_2504"], paths["index_698"]]:
    with open(path) as f:
        header = None
        for line in f:
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header = line.lstrip("#").strip().split("\t")
                break
            raise ValueError(f"{path.name}: hit data line before finding header")

        if header is None:
            raise ValueError(f"{path.name}: no header line found")

        header_cols_stripped = [h.strip() for h in header]
        col = {name: idx for idx, name in enumerate(header_cols_stripped)}
        rows = [ln.rstrip("\n").split("\t") for ln in f if ln.strip()]

    print(f"\n{path.name}: {len(rows)} rows, checking .crai existence...")

    i = 0
    for row in rows:
        i += 1
        sid       = row[col["SAMPLE_NAME"]].strip()
        cram_url  = row[col["ENA_FILE_PATH"]].strip().replace("ftp://", "https://", 1)
        crai_url  = cram_url + ".crai"
        cram_md5  = row[col["MD5SUM"]].strip()

        '''
        req = urllib.request.Request(crai_url, method="HEAD")
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                if resp.status != 200:
                    raise ValueError(f"CRAI not OK for {sid}: HTTP {resp.status} at {crai_url}")
        except urllib.error.HTTPError as e:
            raise ValueError(f"CRAI missing for {sid}: HTTP {e.code} at {crai_url}") from e
        except urllib.error.URLError as e:
            raise ValueError(f"CRAI unreachable for {sid}: {e.reason} at {crai_url}") from e
        '''

        new_entry = {
            "sample_id": sid,
            "cram_url":  cram_url,
            "crai_url":  crai_url,
            "cram_md5":  cram_md5,
        }

        if sid in cram_index:
            raise ValueError(
                f"Conflicting entries for {sid}:\n  existing: {cram_index[sid]}\n  new:      {new_entry}"
            )

        cram_index[sid] = new_entry

        if i % 25 == 0:
            print(f"  {i}/{len(rows)} checked")

    print(f"  Done. Running total: {len(cram_index)}")

print(f"\nCombined: {len(cram_index)} unique samples (expected 3202)")
print(f"Exact duplicates skipped: {n_duplicates}")
print(f"Sample dict: {cram_index.get("HG00443")}")

In [18]:
unique_samples = set()

for key, trio in trios.items():
    for role, member in trio["members"].items():
        sid = member["sample_id"]
        if sid not in cram_index:
            raise ValueError(f"Sample {sid} ({role} in {key}) not found in cram_index")
        info = cram_index[sid]
        member["cram_url"] = info["cram_url"]
        member["crai_url"] = info["crai_url"]
        member["cram_md5"] = info["cram_md5"]
        unique_samples.add(sid)

print(f"Enriched {len(trios)} trios with CRAM URLs")
print(f"Unique samples: {len(unique_samples)}")

Enriched 602 trios with CRAM URLs
Unique samples: 1793


In [20]:
import json
from datetime import datetime, timezone

output = {
    "metadata": {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "n_trios": len(trios),
        "n_unique_samples": len(unique_samples),
        "unique_samples": sorted(unique_samples),
        "n_duos_skipped": n_duos,
        "source_files": {
            "pedigree":   str(paths["pedigree"]),
            "index_2504": str(paths["index_2504"]),
            "index_698":  str(paths["index_698"]),
        },
    },
    "families": trios,
}

out_path = DATA_DIR / "families.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Wrote {out_path} ({out_path.stat().st_size:,} bytes)")
print(f"  Trios:        {output['metadata']['n_trios']}")
print(f"  Unique Samples:  {output['metadata']['n_unique_samples']}")
print(f"  Duos skipped: {output['metadata']['n_duos_skipped']}")

Wrote families.json (782,888 bytes)
  Trios:        602
  Unique Samples:  1793
  Duos skipped: 10
